In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.discriminant_analysis import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score



### Lectura de datos usando escalado (-1,1)

In [2]:
#Ejecutar para cargar datos y escalarlos:

df = pd.read_csv('csv/radiomicas_combinadas_normalizado.csv')
df["Patient"] = df["Patient"].str.replace('"','',regex=False)
df.head(5)

,Patient,original_firstorder_10Percentile,original_firstorder_90Percentile,original_firstorder_Energy,original_firstorder_Entropy,original_firstorder_InterquartileRange,original_firstorder_Kurtosis,original_firstorder_Maximum,original_firstorder_Mean,original_firstorder_MeanAbsoluteDeviation,...,wavelet-LLL_glszm_SmallAreaHighGrayLevelEmphasis,wavelet-LLL_glszm_SmallAreaLowGrayLevelEmphasis,wavelet-LLL_glszm_ZoneEntropy,wavelet-LLL_glszm_ZonePercentage,wavelet-LLL_glszm_ZoneVariance,wavelet-LLL_ngtdm_Busyness,wavelet-LLL_ngtdm_Coarseness,wavelet-LLL_ngtdm_Complexity,wavelet-LLL_ngtdm_Contrast,wavelet-LLL_ngtdm_Strength
0,LUNG1-001,0.264279,-0.471885,-0.279790,0.025573,-0.868932,-0.822722,-0.349502,0.348139,-0.353586,...,0.304767,-0.990892,0.351123,-0.272728,-0.981889,-0.517778,-0.993394,-0.385509,-0.927018,-0.961000
1,LUNG1-002,0.891578,-0.296446,-0.959418,-0.441895,-0.934466,-0.033895,-0.899003,0.711427,-0.869634,...,-0.186468,-0.967982,-0.024807,-0.681319,-0.991054,-0.731196,-0.971461,-0.908586,-0.990167,-0.882556
2,LUNG1-003,0.856728,-0.269456,-0.951453,-0.232302,-0.902913,-0.333322,-0.426578,0.693127,-0.817416,...,0.264072,-0.979473,0.067602,-0.494151,-0.996761,-0.867627,-0.972117,-0.684509,-0.993100,-0.769095
3,LUNG1-004,0.614714,-0.386415,-0.869261,-0.007804,-0.893204,-0.698773,-0.818605,0.464850,-0.517251,...,0.334035,-0.936120,0.444024,-0.353243,-0.998532,-0.911518,-0.953263,-0.681503,-0.939924,-0.699169
4,LUNG1-005,-0.165537,-0.368421,-0.509448,0.272623,-0.771845,-0.875361,-0.887043,0.234176,-0.056650,...,0.340667,-0.946682,0.520499,-0.048338,-0.999756,-0.810449,-0.974874,-0.515473,-0.778467,-0.875750


### Lectura de datos usando estadarizado al z-value

In [6]:
#Ejecutar para cargar datos y estandarizárlos:

df = pd.read_csv('csv/radiomicas_combinadas.csv')
df["Patient"] = df["Patient"].str.replace('"','',regex=False)

df_scaled = df.copy()
df_scaled.iloc[:, 1:] = StandardScaler().fit_transform(df_scaled.iloc[:, 1:])
df = df_scaled
df.head(5)

,Patient,original_firstorder_10Percentile,original_firstorder_90Percentile,original_firstorder_Energy,original_firstorder_Entropy,original_firstorder_InterquartileRange,original_firstorder_Kurtosis,original_firstorder_Maximum,original_firstorder_Mean,original_firstorder_MeanAbsoluteDeviation,...,wavelet-LLL_glszm_SmallAreaHighGrayLevelEmphasis,wavelet-LLL_glszm_SmallAreaLowGrayLevelEmphasis,wavelet-LLL_glszm_ZoneEntropy,wavelet-LLL_glszm_ZonePercentage,wavelet-LLL_glszm_ZoneVariance,wavelet-LLL_ngtdm_Busyness,wavelet-LLL_ngtdm_Coarseness,wavelet-LLL_ngtdm_Complexity,wavelet-LLL_ngtdm_Contrast,wavelet-LLL_ngtdm_Strength
0,LUNG1-001,-0.770901,-0.535339,2.935450,0.491218,-0.139063,-0.912903,1.967610,-0.462555,0.662133,...,0.647232,-0.493910,0.703885,0.273290,0.051056,1.702740,-0.385697,1.970209,-0.113225,-1.087242
1,LUNG1-002,0.747366,0.936893,-0.456281,-0.794639,-0.390801,1.400323,-0.617684,0.976466,-0.806721,...,-0.653012,-0.332510,-0.507615,-0.920339,-0.037666,0.500301,-0.256509,-0.900346,-0.454782,-0.665691
2,LUNG1-003,0.663018,1.163391,-0.416535,-0.218115,-0.269594,0.522257,1.604981,0.903976,-0.658090,...,0.539516,-0.413458,-0.209812,-0.373560,-0.092904,-0.268380,-0.260377,0.329351,-0.470647,-0.055960
3,LUNG1-004,0.077266,0.181902,-0.006347,0.399409,-0.232299,-0.549426,-0.239424,-0.000251,0.196286,...,0.724701,-0.108041,1.003277,0.038079,-0.110048,-0.515669,-0.149322,0.345848,-0.183034,0.319816
4,LUNG1-005,-1.811196,0.332901,1.789323,1.170775,0.233883,-1.067267,-0.561414,-0.913973,1.507316,...,0.742255,-0.182447,1.249730,0.928806,-0.121896,0.053771,-0.276614,1.256989,0.690245,-0.629115


### Lectura de datos de clasificaciones de pacientes

In [3]:
df2 = pd.read_csv('csv/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv')
df2.head(5)

,PatientID,age,clinical.T.Stage,Clinical.N.Stage,Clinical.M.Stage,Overall.Stage,Histology,gender,Survival.time,deadstatus.event
0,LUNG1-001,78.7515,2.0,3,0,IIIb,large cell,male,2165,1
1,LUNG1-002,83.8001,2.0,0,0,I,squamous cell carcinoma,male,155,1
2,LUNG1-003,68.1807,2.0,3,0,IIIb,large cell,male,256,1
3,LUNG1-004,70.8802,2.0,1,0,II,squamous cell carcinoma,male,141,1
4,LUNG1-005,80.4819,4.0,2,0,IIIb,squamous cell carcinoma,male,353,1


In [4]:
df2 = df2.dropna(subset=["Overall.Stage"])  # Drop rows with NaN in Overall.Stage column

df2 = df2[df2["PatientID"].isin(df["Patient"])]  # Keep only patients in df
df = df[df["Patient"].isin(df2["PatientID"])]  # Keep only patients in df2
print(df.shape)
print(df2.shape)


(417, 852)
(417, 10)


### Selección de características 

In [5]:
SELECTOR = "stable" #all, stable, firstorder, glcm, gldm, glrlm, glszm, ngtdm, shape

stable_features = pd.read_csv('csv/mejores_radiomicas.csv').iloc[:-1, 0].values

switcher = {
    "all": (df.columns[1:], "Seleccionadas todas las caractrísticas."),
    "stable": (stable_features, "Seleccionadas características estables."),
    "firstorder": (df.filter(like='original_firstorder').columns, "Seleccionadas características de primer orden."),
    "glcm": (df.filter(like='original_glcm').columns, "Seleccionadas características de GLCM."),
    "gldm": (df.filter(like='original_gldm').columns, "Seleccionadas características de GLDM."),
    "glrlm": (df.filter(like='original_glrlm').columns, "Seleccionadas características de GLRLM."),
    "glszm": (df.filter(like='original_glszm').columns, "Seleccionadas características de GLSZM."),
    "ngtdm": (df.filter(like='original_ngtdm').columns, "Seleccionadas características de NGTDM."),
    "shape": (df.filter(like='original_shape').columns, "Seleccionadas características de forma.")
}


features = switcher.get(SELECTOR, (df.columns[1:], "Seleccionadas todas las caractrísticas."))[0]
print(switcher.get(SELECTOR, (df.columns[1:], "Seleccionadas todas las caractrísticas."))[1])
print(features)

Seleccionadas características estables.
<StringArray>
[                        'original_firstorder_Energy',
                           'original_firstorder_Mean',
                         'original_firstorder_Median',
                'original_firstorder_RootMeanSquared',
                    'original_firstorder_TotalEnergy',
                    'original_glcm_DifferenceAverage',
                                   'original_glcm_Id',
                                  'original_glcm_Idm',
                                 'original_glcm_Idmn',
              'original_gldm_DependenceNonUniformity',
 ...
 'wavelet-LLL_glrlm_RunLengthNonUniformityNormalized',
                    'wavelet-LLL_glrlm_RunPercentage',
                      'wavelet-LLL_glrlm_RunVariance',
                 'wavelet-LLL_glrlm_ShortRunEmphasis',
            'wavelet-LLL_glszm_SizeZoneNonUniformity',
  'wavelet-LLL_glszm_SizeZoneNonUniformityNormalized',
                'wavelet-LLL_glszm_SmallAreaEmphasis',
      

### Random Forest for Overall stage

In [6]:
classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Build aligned dataset (X, y) without NaNs
tmp = df[["Patient"] + list(features)].copy()
tmp = tmp.dropna(subset=features)

stage_map = df2.set_index("PatientID")["Overall.Stage"]
tmp["Overall.Stage"] = tmp["Patient"].map(stage_map)
tmp = tmp.dropna(subset=["Overall.Stage"])

X = tmp[list(features)]
y = tmp["Overall.Stage"]

# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Out-of-fold predictions for full report
y_pred_cv = cross_val_predict(classifier, X, y, cv=cv)

# Optional scalar metric across folds
f1_scores = cross_val_score(classifier, X, y, cv=cv, scoring="f1_weighted")

print(f"Weighted F1 (5-fold CV): {f1_scores.mean():.4f} ± {f1_scores.std():.4f}")
print(classification_report(y, y_pred_cv))
print(confusion_matrix(y, y_pred_cv))

Weighted F1 (5-fold CV): 0.3344 ± 0.0226
              precision    recall  f1-score   support

           I       0.40      0.24      0.30        92
          II       0.00      0.00      0.00        40
        IIIa       0.22      0.15      0.17       110
        IIIb       0.43      0.70      0.53       175

    accuracy                           0.39       417
   macro avg       0.26      0.27      0.25       417
weighted avg       0.33      0.39      0.34       417

[[ 22   0  13  57]
 [  4   0   9  27]
 [ 13   0  16  81]
 [ 16   1  35 123]]


### Random Forest for Hystology

In [ ]:
# Build aligned dataset (X, y) without NaNs
tmp = df[["Patient"] + list(features)].copy()
tmp = tmp.dropna(subset=features)

stage_map = df2.set_index("PatientID")["Histology"]
tmp["Histology"] = tmp["Patient"].map(stage_map)
tmp = tmp.dropna(subset=["Histology"])

X = tmp[list(features)]
y = tmp["Histology"]

# 5-fold stratified cross-validation
classifier = RandomForestClassifier(n_estimators=100, random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Out-of-fold predictions for full report
y_pred_cv = cross_val_predict(classifier, X, y, cv=cv)

# Optional scalar metric across folds
f1_scores = cross_val_score(classifier, X, y, cv=cv, scoring="f1_weighted")

print(f"Weighted F1 (5-fold CV): {f1_scores.mean():.4f} ± {f1_scores.std():.4f}")
print(classification_report(y, y_pred_cv))
print(confusion_matrix(y, y_pred_cv))

Weighted F1 (5-fold CV): 0.3466 ± 0.0596
                         precision    recall  f1-score   support

         adenocarcinoma       0.00      0.00      0.00        50
             large cell       0.35      0.39      0.37       111
                    nos       0.23      0.08      0.12        62
squamous cell carcinoma       0.46      0.67      0.54       152

               accuracy                           0.40       375
              macro avg       0.26      0.28      0.26       375
           weighted avg       0.33      0.40      0.35       375

[[  0  19   3  28]
 [  4  43   6  58]
 [  0  21   5  36]
 [  3  39   8 102]]


### MDA sum

In [8]:
def compute_mda_sum(estimator, X, y, cv, n_repeats=10, random_state=42):
    mda_by_fold = []
    all_importances = []

    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model = clone(estimator)
        model.fit(X_train, y_train)

        baseline_accuracy = accuracy_score(y_test, model.predict(X_test))
        perm = permutation_importance(
            model,
            X_test,
            y_test,
            scoring="accuracy",
            n_repeats=n_repeats,
            random_state=random_state + fold_idx,
            n_jobs=-1,
        )

        fold_result = pd.DataFrame({
            "feature": X.columns,
            "fold": fold_idx,
            "baseline_accuracy": baseline_accuracy,
            "mda_sum_fold": perm.importances.sum(axis=1),
            "mda_mean_fold": perm.importances_mean,
            "mda_std_fold": perm.importances_std,
        })

        mda_by_fold.append(fold_result)
        all_importances.append(perm.importances)

    mda_by_fold = pd.concat(mda_by_fold, ignore_index=True)
    all_importances = np.concatenate(all_importances, axis=1)

    ranking = (
        mda_by_fold.groupby("feature", as_index=False)
        .agg(
            mda_sum=("mda_sum_fold", "sum"),
            mda_mean=("mda_mean_fold", "mean"),
            mda_std=("mda_mean_fold", "std"),
        )
        .sort_values("mda_sum", ascending=False)
        .reset_index(drop=True)
    )

    ranking["times_more_important"] = ranking["mda_sum"] / ranking["mda_sum"].max()

    return ranking, mda_by_fold, all_importances




#### Overall Stage

In [11]:
tmp = df[["Patient"] + list(features)].copy()
tmp = tmp.dropna(subset=features)

stage_map = df2.set_index("PatientID")["Overall.Stage"]
tmp["Overall.Stage"] = tmp["Patient"].map(stage_map)
tmp = tmp.dropna(subset=["Overall.Stage"])

X = tmp[list(features)]
y = tmp["Overall.Stage"]

classifier = RandomForestClassifier(n_estimators=100, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

stage_mda_ranking, mda_by_fold, raw_importances = compute_mda_sum(
    classifier, X, y, cv=cv, n_repeats=20, random_state=42
    )


print("Top 20 features by MDA sum for Overall.Stage:")
display(stage_mda_ranking.head(20))

Top 20 features by MDA sum for Overall.Stage:


,feature,mda_sum,mda_mean,mda_std,times_more_important
0,wavelet-HHH_glrlm_LongRunHighGrayLevelEmphasis,1.144291,0.011443,0.010455,1.000000
1,wavelet-HHH_ngtdm_Strength,1.143574,0.011436,0.013302,0.999373
2,wavelet-LLH_glcm_Contrast,1.072146,0.010721,0.013085,0.936952
3,wavelet-HHL_glcm_ClusterProminence,0.961704,0.009617,0.010542,0.840436
4,wavelet-LHH_glrlm_LongRunLowGrayLevelEmphasis,0.961130,0.009611,0.007407,0.839935
5,wavelet-HLH_gldm_GrayLevelVariance,0.951807,0.009518,0.014111,0.831787
6,wavelet-LHL_glszm_SmallAreaLowGrayLevelEmphasis,0.886690,0.008867,0.004633,0.774881
7,wavelet-LLL_glrlm_LongRunHighGrayLevelEmphasis,0.867183,0.008672,0.009312,0.757834
8,wavelet-HLL_firstorder_Mean,0.854561,0.008546,0.008995,0.746804
9,wavelet-HHH_firstorder_Maximum,0.852697,0.008527,0.008164,0.745174


#### Histology

In [12]:
tmp = df[["Patient"] + list(features)].copy()
tmp = tmp.dropna(subset=features)

stage_map = df2.set_index("PatientID")["Histology"]
tmp["Histology"] = tmp["Patient"].map(stage_map)
tmp = tmp.dropna(subset=["Histology"])

X = tmp[list(features)]
y = tmp["Histology"]

classifier = RandomForestClassifier(n_estimators=100, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

histology_mda_ranking, mda_by_fold, raw_importances = compute_mda_sum(
    classifier, X, y, cv=cv, n_repeats=20, random_state=42
    )


print("Top 20 features by MDA sum for histology:")
display(histology_mda_ranking.head(20))

Top 20 features by MDA sum for histology:


,feature,mda_sum,mda_mean,mda_std,times_more_important
0,original_gldm_SmallDependenceEmphasis,0.680000,0.006800,0.010545,1.000000
1,wavelet-HHL_gldm_LargeDependenceEmphasis,0.680000,0.006800,0.009456,1.000000
2,wavelet-HHH_glcm_SumAverage,0.666667,0.006667,0.009189,0.980392
3,wavelet-LHL_glcm_DifferenceEntropy,0.640000,0.006400,0.014225,0.941176
4,wavelet-HHH_firstorder_Variance,0.626667,0.006267,0.012235,0.921569
5,wavelet-HHL_ngtdm_Busyness,0.613333,0.006133,0.011675,0.901961
6,wavelet-LHL_glcm_JointEntropy,0.586667,0.005867,0.005684,0.862745
7,wavelet-HLL_firstorder_Mean,0.560000,0.005600,0.010117,0.823529
8,wavelet-HHL_gldm_LargeDependenceLowGrayLevelEm...,0.546667,0.005467,0.008385,0.803922
9,wavelet-LHL_gldm_LargeDependenceEmphasis,0.533333,0.005333,0.007717,0.784314


### Retrainign with top features:

#### Overall Stage

In [15]:
features_to_keep = stage_mda_ranking.head(20)["feature"].values
tmp = df[["Patient"] + list(features_to_keep)].copy()
tmp = tmp.dropna(subset=features_to_keep)

stage_map = df2.set_index("PatientID")["Overall.Stage"]
tmp["Overall.Stage"] = tmp["Patient"].map(stage_map)
tmp = tmp.dropna(subset=["Overall.Stage"])

X = tmp[list(features_to_keep)]
y = tmp["Overall.Stage"]

# 5-fold stratified cross-validation
classifier = RandomForestClassifier(n_estimators=100, random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Out-of-fold predictions for full report
y_pred_cv = cross_val_predict(classifier, X, y, cv=cv)

# Optional scalar metric across folds
f1_scores = cross_val_score(classifier, X, y, cv=cv, scoring="f1_weighted")

print(f"Weighted F1 (5-fold CV): {f1_scores.mean():.4f} ± {f1_scores.std():.4f}")
print(classification_report(y, y_pred_cv))
print(confusion_matrix(y, y_pred_cv))

Weighted F1 (5-fold CV): 0.3696 ± 0.0516
              precision    recall  f1-score   support

           I       0.31      0.22      0.26        92
          II       0.00      0.00      0.00        40
        IIIa       0.32      0.22      0.26       110
        IIIb       0.48      0.75      0.59       175

    accuracy                           0.42       417
   macro avg       0.28      0.30      0.28       417
weighted avg       0.35      0.42      0.37       417

[[ 20   0  16  56]
 [  9   0  12  19]
 [ 17   1  24  68]
 [ 18   2  23 132]]


#### Histology

In [16]:
features_to_keep = stage_mda_ranking.head(20)["feature"].values
tmp = df[["Patient"] + list(features_to_keep)].copy()
tmp = tmp.dropna(subset=features_to_keep)

stage_map = df2.set_index("PatientID")["Histology"]
tmp["Histology"] = tmp["Patient"].map(stage_map)
tmp = tmp.dropna(subset=["Histology"])

X = tmp[list(features_to_keep)]
y = tmp["Histology"]

# 5-fold stratified cross-validation
classifier = RandomForestClassifier(n_estimators=100, random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Out-of-fold predictions for full report
y_pred_cv = cross_val_predict(classifier, X, y, cv=cv)

# Optional scalar metric across folds
f1_scores = cross_val_score(classifier, X, y, cv=cv, scoring="f1_weighted")

print(f"Weighted F1 (5-fold CV): {f1_scores.mean():.4f} ± {f1_scores.std():.4f}")
print(classification_report(y, y_pred_cv))
print(confusion_matrix(y, y_pred_cv))

Weighted F1 (5-fold CV): 0.3328 ± 0.0095
                         precision    recall  f1-score   support

         adenocarcinoma       0.00      0.00      0.00        50
             large cell       0.34      0.35      0.34       111
                    nos       0.16      0.06      0.09        62
squamous cell carcinoma       0.45      0.67      0.54       152

               accuracy                           0.39       375
              macro avg       0.24      0.27      0.24       375
           weighted avg       0.31      0.39      0.33       375

[[  0  15   3  32]
 [  3  39  10  59]
 [  0  23   4  35]
 [  3  39   8 102]]
